In [1]:
from utils import *
from CS_feature_extractor import *
from CS_based_early_stopping import *

/home/guangya/SC_conf/src/utils.py:376: SyntaxWarning: invalid escape sequence '\%'
  string = string.replace("\%", "")
[nltk_data] Downloading package punkt to /home/guangya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/guangya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# How to run the code to get Acc and # of API calls

In [2]:
DATA_DIR = "../data/Evaluation_CoTs/"
input_file_path = os.path.join(DATA_DIR, 'GSM8K_GPT4o_mini.csv')
df_llama = pd.read_csv(input_file_path).reset_index(drop=True)

In [3]:
df_test = df_llama.sample(100).reset_index(drop=True) # get partial data for test

In [4]:
df_test  # Please put your question and CoT in the given format

,Name,Category,Question,Correct Answer,CoT_0,Final Answer_0,Instruction Violation_0,CoT_1,Final Answer_1,Instruction Violation_1,...,CoT_37,Final Answer_37,Instruction Violation_37,CoT_38,Final Answer_38,Instruction Violation_38,CoT_39,Final Answer_39,Instruction Violation_39,Prompt_File
0,GSM8K_test,Math,Cars have lined up on the motorway. Some of th...,5,Step 1: Identify the given information - There...,5,"[(0, 0)]",Step 1: Identify the given information - There...,insufficient information,"[(0, 0)]",...,Step 1: Identify the given information - There...,5,"[(0, 0)]",Step 1: Identify the given information - Total...,5,"[(0, 0)]",Step 1: Identify the given information - There...,5,"[(0, 0)]",few_CoT.json
1,GSM8K_test,Math,Charlie wants to sell beeswax candles. For ev...,20,Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Charl...,$20.00,"[(0, 0)]",...,Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Charl...,$20.00,"[(0, 0)]",Step 1: Identify the given information - Charl...,20.00,"[(0, 0)]",few_CoT.json
2,GSM8K_test,Math,Eliza's rate per hour for the first 40 hours s...,460,Step 1: Identify the given information - Eliza...,460,"[(0, 0)]",Step 1: Identify the given information - Eliza...,460,"[(0, 0)]",...,Step 1: Identify the given information - Regul...,460,"[(0, 0)]",Step 1: Identify the given information - Eliza...,$460,"[(0, 0)]",Step 1: Identify the given information - Eliza...,460,"[(0, 0)]",few_CoT.json
3,GSM8K_test,Math,Peter plans to go to the movies this week. He ...,3,Step 1: Identify the costs involved in going t...,3,"[(0, 0)]",Step 1: Identify the costs involved in going t...,3,"[(0, 0)]",...,Step 1: Identify the costs involved in going t...,3,"[(0, 0)]",Step 1: Identify the costs involved - The cost...,3,"[(0, 0)]",Step 1: Identify the costs associated with goi...,3,"[(0, 0)]",few_CoT.json
4,GSM8K_test,Math,Jan has three times the number of pets as Marc...,28,Step 1: Identify the given information - Cindy...,28,"[(0, 0)]",Step 1: Identify the given information - Cindy...,28,"[(0, 0)]",...,Step 1: Identify the number of pets Cindy has....,28,"[(0, 0)]",Step 1: Identify the number of pets Cindy has....,28,"[(0, 0)]",Step 1: Identify the number of pets Cindy has....,28,"[(0, 0)]",few_CoT.json
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,GSM8K_test,Math,"Mary is an avid gardener. Yesterday, she recei...",58,Step 1: Identify the given information - Mary ...,58,"[(0, 0)]",Step 1: Identify the given information - Mary ...,58,"[(0, 0)]",...,Step 1: Identify the given information - Mary ...,58,"[(0, 0)]",Step 1: Identify the given information - Mary ...,58,"[(0, 0)]",Step 1: Identify the given information - Mary ...,58,"[(0, 0)]",few_CoT.json
96,GSM8K_test,Math,Jim spends 2 hours watching TV and then decide...,36,Step 1: Identify the given information - Jim s...,36 hours,"[(0, 0)]",Step 1: Identify the given information - Jim s...,36 hours,"[(0, 0)]",...,Step 1: Identify the time Jim spends watching ...,36 hours,"[(0, 0)]",Step 1: Identify the time Jim spends on each a...,36 hours,"[(0, 0)]",Step 1: Identify the time spent on each activi...,36 hours,"[(0, 0)]",few_CoT.json
97,GSM8K_test,Math,Vincent can buy flowers in packages of 3 for $...,6,Step 1: Identify the two pricing options for b...,6,"[(0, 0)]",Step 1: Identify the two options for buying fl...,6,"[(0, 0)]",...,Step 1: Identify the cost of each package type...,6,"[(0, 0)]",Step 1: Identify the two options for buying fl...,$6.00,"[(0, 0)]",Step 1: Identify the given information - Vince...,$6.00,"[(0, 0)]",few_CoT.json
98,GSM8K_test,Math,A football team played 22 games. They won 8 mo...,15,Step 1: Identify the given information - Total...,15,"[(0, 0)]",Step 1: Identify the given information - Total...,15,"[(0, 0)]",...,Step 1: Identify the total number of games pla...,15,"[(0, 0)]",Step 1: Identify the given

In [5]:
feature_li = ['LEN', 'QUA_IM', 'DIF_IV', 'SIM_COT_BIGRAM', 'SIM_COT_AGG', 'SIM_AC_BIGRAM', 'SIM_AC_AGG', 'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE'] 
# This includes total 10 features introduced in the paper; please see extract features for more details
data = extract_feature(df_test,feature_li)

jaccard with bigram time cost: 3.353882312774658s
jaccard with aggregation time cost: 32.11609435081482s


100%|██████████| 100/100 [00:02<00:00, 37.19it/s]


In [10]:
pd.DataFrame(data).head(5) # data is saved in json format

,id,Name,correct answer,CoT answers,Correctness,QUA_IM,SIM_COT_BIGRAM,SIM_INPUT,SIM_AC_BIGRAM,LEN,STEP_COHERENCE,DIF_IV,STEP_COUNT,SIM_AC_AGG,SIM_COT_AGG
0,0,GSM8K_test,5,"[5.0, insufficient information, 5.0, 5.0, 5.0,...","[1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.42105263157894735, 0.6057692307692308,...","[0.3176470588235294, 0.3465346534653465, 0.376...","[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.29916913715088156, 0.24260394313373215, 0.3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 7, 6, 7, 6, 7, 8, 7, 6, 6, 6, 6, 8, ...","[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.42105263157894735, 0.540983606557377, ..."
1,1,GSM8K_test,20,"[20.0, $20.00, $20.00, $20.00, $20.00, 20.0, $...","[1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.6883116883116883, 0.7236842105263157, ...","[0.3835616438356164, 0.38961038961038963, 0.47...","[0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.3545866935483871, 0.3089693681125906, 0.298...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.6883116883116883, 0.7283950617283951, ..."
2,2,GSM8K_test,460,"[460.0, 460.0, 460.0, 460.0, 460.0, 460.0, $46...","[1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8163265306122449, 0.6923076923076923, ...","[0.35, 0.3650793650793651, 0.2698412698412699,...","[0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[0.4056964573268921, 0.3460942760942761, 0.480...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 6, 5, 6, 5, 6, 6, 6, 5, 5, 6, 6, 5, 4, 6, ...","[0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, ...","[0.0, 0.8163265306122449, 0.6666666666666667, ..."
3,3,GSM8K_test,3,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.5747126436781609, 0.5764705882352941, ...","[0.27142857142857146, 0.2325581395348837, 0.24...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, ...","[0.24815323565323566, 0.19801708072821905, 0.3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 4, 4, 5, 5, 5, 5, 5, 4, 5, 4, 5, 5, 4, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.5747126436781609, 0.6363636363636364, ..."
4,4,GSM8K_test,28,"[28.0, 28.0, 28.0, 28.0, 28.0, 28.0, 28.0, 28....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.7857142857142857, 0.8095238095238095, ...","[0.33333333333333337, 0.40476190476190477, 0.3...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.443019943019943, 0.4901852533431481, 0.4133...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.7857142857142857, 0.9534883720930233, ..."


In [11]:
df_processed = pd.DataFrame(data)

In [12]:
df_processed = calculate_SC_correctness(df_processed)

# Calculate Early Stopping Correctness with a specific window size
window_size = 5  # Define your window size
df_processed = calculate_ES_correctness(df_processed, window_size)

# Calculate Adaptive Consensus Correctness
df_processed = calculate_ASC_correctness(df_processed)

ES execution time: 0.0027 seconds
ASC execution time: 0.0211 seconds


In [13]:
df_processed.head() 

,id,Name,correct answer,CoT answers,Correctness,QUA_IM,SIM_COT_BIGRAM,SIM_INPUT,SIM_AC_BIGRAM,LEN,STEP_COHERENCE,DIF_IV,STEP_COUNT,SIM_AC_AGG,SIM_COT_AGG,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps
0,0,GSM8K_test,5,"[5.0, insufficient information, 5.0, 5.0, 5.0,...","[1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.42105263157894735, 0.6057692307692308,...","[0.3176470588235294, 0.3465346534653465, 0.376...","[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.29916913715088156, 0.24260394313373215, 0.3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 7, 6, 7, 6, 7, 8, 7, 6, 6, 6, 6, 8, ...","[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.42105263157894735, 0.540983606557377, ...",1,1,7,1,7
1,1,GSM8K_test,20,"[20.0, $20.00, $20.00, $20.00, $20.00, 20.0, $...","[1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.6883116883116883, 0.7236842105263157, ...","[0.3835616438356164, 0.38961038961038963, 0.47...","[0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.3545866935483871, 0.3089693681125906, 0.298...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.6883116883116883, 0.7283950617283951, ...",0,0,13,0,12
2,2,GSM8K_test,460,"[460.0, 460.0, 460.0, 460.0, 460.0, 460.0, $46...","[1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8163265306122449, 0.6923076923076923, ...","[0.35, 0.3650793650793651, 0.2698412698412699,...","[0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[0.4056964573268921, 0.3460942760942761, 0.480...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 6, 5, 6, 5, 6, 6, 6, 5, 5, 6, 6, 5, 4, 6, ...","[0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, ...","[0.0, 0.8163265306122449, 0.6666666666666667, ...",1,1,5,1,4
3,3,GSM8K_test,3,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.5747126436781609, 0.5764705882352941, ...","[0.27142857142857146, 0.2325581395348837, 0.24...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, ...","[0.24815323565323566, 0.19801708072821905, 0.3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[5, 5, 4, 4, 5, 5, 5, 5, 5, 4, 5, 4, 5, 5, 4, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.5747126436781609, 0.6363636363636364, ...",1,1,5,1,4
4,4,GSM8K_test,28,"[28.0, 28.0, 28.0, 28.0, 28.0, 28.0, 28.0, 28....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.7857142857142857, 0.8095238095238095, ...","[0.33333333333333337, 0.40476190476190477, 0.3...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.443019943019943, 0.4901852533431481, 0.4133...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.7857142857142857, 0.9534883720930233, ...",1,1,5,1,4


In [14]:
# TO DO 1: Demonstrate 1st way of doing without customize model; 
# 2: demonstrate how to do it with customized model;
# 4: Test code to run CoT
# 5: Write doc on how to run the python file. (special explanations for feature extraction)

In [15]:
feature_li = ['LEN', 'QUA_IM', 'DIF_IV', 'SIM_COT_BIGRAM', 'SIM_AC_BIGRAM',  'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE'] # We can take less if for test
df_confidence_scores = trained_LR_model(df_processed, feature_li, report_auroc=False) # Note that we keep the test data only to aviod overfit

KeyError: 'Model'

In [25]:
df_llama_confidence_scores.head()

,id,Name,Model,correct answer,CoT answers,Correctness,MATH_TERM_DENSITY,STEP_COHERENCE,SIM_AC_AGG,SIM_COT_AGG,...,DIF_IV,QUA_IM,STEP_COUNT,SIM_AC_BIGRAM,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps,confidence_score
0,40,BigBench_easy,llama3,C,"[D, C, A, E, E, C, E, A, D, A, C, E, E, D, D, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0.5, 0.37209302325581395, 0.45033112582781454...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...",0,0,40,0,40,"[0.23638707935418551, 0.10582059028380454, 0.1..."
1,7,MathQA_challenge_test,llama3,c,"[E, C, E, E, E, E, E, E, C, C, D, D, E, B, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.012269938650306749, 0.02030456852791878, 0....","[0, 0, 0, 0, 0.32393939393939397, 0, 0, 0.2148...","[0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, ...","[0.5, 0.3652173913043478, 0.38, 0.254545454545...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 1, 3, 0, 0, 5, 0, 0, 0, 0, 0, 0, 3, ...","[0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, ...",0,0,7,0,7,"[0.18175323341013105, 0.06319644962629163, 0.1..."
2,10,MathQA_dev,llama3,b,"[E, A, A, E, E, E, E, E, E, E, A, E, E, B, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.005050505050505051, 0.011764705882352941, 0...","[0.2361111111111111, 0.22580645161290322, 0, 0...","[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, ...","[0.5, 0.6931818181818181, 0.6274509803921569, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 0, 0, 0, 2, 0, 0, 0, 2, 2, 2, 0, 2, 0, ...","[0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, ...",0,0,8,0,10,"[0.18299950629311443, 0.41321052181956053, 0.4..."
3,44,MathQA_challenge_test,llama3,a,"[E, E, E, D, E, E, E, E, E, E, E, E, E, E, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.011764705882352941, 0.011494252873563218, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0.32205919503079744, ...","[0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.5, 0.38497652582159625, 0.296137339055794, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,9,0,7,"[0.17491185964273384, 0.18835863334431088, 0.1..."
4,33,BigBench_easy,llama3,D,"[B, E, C, A, C, C, C, E, A, E, C, A, E, E, C, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.5, 0.45112781954887216, 0.3549382716049383,...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, ...",0,0,40,0,40,"[0.24724127099954366, 0.18596011704801862, 0.0..."


In [26]:
N = 5
threshold = 0.5

# Applying early stopping mechanism
df_final = CS_early_stopping(df=df_llama_confidence_scores, threshold=threshold, N=N)

SC_ACC : 0.2
ES_ACC : 0.2
CS_ACC : 0.3333333333333333
SC_Avg_Steps : 40
ES_Avg_Steps : 21.6
CS_Avg_Steps : 34.266666666666666
ASC_Avg_Steps : 18.066666666666666
ASC_ACC : 0.2


# How to Run the Code the get CoTs?